# Grid Search Simulation across 9 Hyperparameter Configurations

This notebook runs the fine-tuning training simulation across 9 hyperparameter configurations in a 3x3 grid pattern.
- Stores **only the final checkpoints** for each configuration.
- Supports  setting (True/False).
- Pushes final checkpoints to Kaggle dataset using .

In [ ]:
import os
import sys
sys.path.append('../utils')
sys.path.append('./src/utils')

# --- CONFIGURABLE SIMULATION PARAMETERS ---
USE_WIREFRAMES = False               # Set to True to include wireframe screen images
SAVE_INTERMEDIATE_CHECKPOINTS = False # Set to False to store ONLY final checkpoints
# Select specific grid index pairs [(i, j)] to run, or None to execute all 9 configurations
RUN_GRID_INDICES = None               # e.g., [(0,0), (0,1)] or None for all 9 (0..2 x 0..2)


In [ ]:
from grid_pipeline import (
    get_grid_config,
    get_all_grid_configs,
    create_differential_optimizer,
    run_fine_tuning_experiment,
    run_grid_search
)
from dataset_utils import push_checkpoints_to_kaggle_dataset, trigger_kaggle_notebook

# Display the 9 configuration parameters in the 3x3 grid
all_configs = get_all_grid_configs()
print(f"=== 9 GRID CONFIGURATIONS DEFINED ===")
for cfg in all_configs:
    print(f"Config [{cfg['grid_i']}, {cfg['grid_j']}]: lr_warmup={cfg['lr_warmup']}, lr_drift={cfg['lr_drift']}, lr_early={cfg['lr_early']}, lr_late={cfg['lr_late']}, lr_head={cfg['lr_head']}")


## Execute Grid Search Simulation
Runs training and validation for the selected grid configurations, logging train/validation losses and saving final checkpoints.

In [ ]:
all_results = run_grid_search(
    grid_indices=RUN_GRID_INDICES,
    use_wireframes=USE_WIREFRAMES,
    save_intermediate_checkpoints=SAVE_INTERMEDIATE_CHECKPOINTS
)
print("
✅ Grid simulation complete for all requested configurations!")


## Push Final Checkpoints to Kaggle Dataset & Trigger Analysis Notebook

In [ ]:
checkpoint_files = []
for tag, res in all_results.items():
    ft_ckpt = res.get('fine_tuning', {}).get('checkpoint')
    if ft_ckpt and os.path.exists(ft_ckpt):
        checkpoint_files.append(ft_ckpt)
    sc_ckpt = res.get('screen_scratch', {}).get('checkpoint')
    if sc_ckpt and os.path.exists(sc_ckpt):
        checkpoint_files.append(sc_ckpt)

print(f"Found {len(checkpoint_files)} final checkpoint(s) to push to Kaggle dataset.")
if checkpoint_files:
    push_checkpoints_to_kaggle_dataset(
        dataset_slug="nazariyyuchnovskiy/20-checkpoint-batchnorm",
        checkpoint_paths=checkpoint_files,
        version_notes="Grid search simulation run final checkpoints"
    )
    trigger_kaggle_notebook(
        notebook_slug="nazariyyuchnovskiy/analysis-model",
        delay_seconds=10
    )
